# Week 2, Model 3: The Neural Network

**A small brain made of maths that spots patterns across many stats at once.**

*Author: The Genius Project Year 3*

---

### What you will build
- A neural network that mixes every stat together to judge a match.
- Win chances and a predicted scoreline for each upcoming fixture.
- A feel for why neural networks are powerful but harder to read than a tree.

### The big idea, in one breath
A neural network is layers of tiny switches. Each switch listens to the stats, and together they learn patterns that a single rule would miss, like how attack, form, and injuries combine.

> This notebook uses a realistic teaching dataset. The numbers are believable and
> the patterns are real, but they are not official statistics. The goal is to learn
> how the models think, not to run a betting shop.

**Football note:** we call the sport *football* (some countries call it soccer).
Every stat is explained in plain words below, so you do not need to be a fan to follow along.

### The stats we use, in plain words

| Column | What it means |
| --- | --- |
| `team` | The national team, for example Brazil or Japan. |
| `confederation` | The region a team belongs to. UEFA is Europe, CONMEBOL is South America, CAF is Africa, AFC is Asia, CONCACAF is North and Central America. |
| `fifa_ranking` | The team's position on the official world ladder. 1 is the best. A smaller number is better. |
| `fifa_points` | The score behind the ranking. More points means a stronger team. |
| `elo_rating` | Another strength score, borrowed from chess. Higher is stronger. |
| `world_cup_titles` | How many times the team has won the World Cup. |
| `avg_goals_scored` | Goals the team scores in a typical match. This is their attack. Higher is better. |
| `avg_goals_conceded` | Goals the team lets in during a typical match. This is their defence. Lower is better. |
| `possession_pct` | The share of the match, out of 100, that the team keeps the ball. More possession usually means more control. |
| `pass_accuracy_pct` | Out of every 100 passes, how many reach a team mate. |
| `shots_per_game` | How many attempts at goal the team takes in a match. |
| `shots_on_target_per_game` | Shots that were heading into the net, whether or not they scored. |
| `big_chances_per_game` | Clear chances to score in a match, the kind a player is expected to finish. |
| `clean_sheet_pct` | Out of 100 matches, how many the team finishes without letting in a single goal. |
| `win_rate_last10` | The share of the last 10 matches the team won, from 0 to 1. 0.7 means they won 7 of 10. |
| `form_points_last5` | Points from the last 5 matches. A win is 3 points, a draw is 1, a loss is 0. The most is 15. |
| `squad_avg_age` | The average age of the players in the squad. |
| `squad_value_million_eur` | The total transfer value of the squad, in millions of euros. A rough measure of talent. |
| `top_league_player_share` | The share of players, from 0 to 1, who play for clubs in the strongest leagues. |
| `set_piece_goal_pct` | Out of 100 goals, how many come from free kicks and corners rather than open play. |
| `key_player_injuries` | How many important players are currently injured. |
| `is_2026_host` | 1 if the team is hosting the 2026 World Cup (USA, Canada, Mexico), otherwise 0. |

You do not have to memorise these. Glance back whenever a column name shows up.

### Step 1: Load the data and build the numbers

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the three tables.
teams = pd.read_csv("data/teams.csv")
matches = pd.read_csv("data/matches.csv")
upcoming = pd.read_csv("data/upcoming_matches.csv")

# 2. The numbers we will feed the model for each team.
#    Think of these as the "stats card" for a team.
TEAM_STATS = [
    "fifa_points", "elo_rating", "avg_goals_scored", "avg_goals_conceded",
    "possession_pct", "pass_accuracy_pct", "shots_on_target_per_game",
    "big_chances_per_game", "clean_sheet_pct", "win_rate_last10",
    "form_points_last5", "squad_value_million_eur", "top_league_player_share",
    "key_player_injuries",
]

# 3. A helper that takes a match (team_a vs team_b) and returns one flat row
#    of numbers: team A's stats, team B's stats, and the gap between them.
stats_by_team = teams.set_index("team")

def build_features(team_a, team_b):
    a = stats_by_team.loc[team_a, TEAM_STATS]
    b = stats_by_team.loc[team_b, TEAM_STATS]
    row = {}
    for stat in TEAM_STATS:
        row[f"{stat}_a"] = a[stat]      # team A value
        row[f"{stat}_b"] = b[stat]      # team B value
        row[f"{stat}_diff"] = a[stat] - b[stat]   # the gap (A minus B)
    return row

# 4. Turn every historical match into a row of features.
feature_rows = [build_features(r.team_a, r.team_b) for r in matches.itertuples()]
X_full = pd.DataFrame(feature_rows)

# 5. For the model we keep the GAP columns (team A value minus team B value).
#    These are the easiest to read: a positive weight on a gap means
#    "when team A has more of this than team B, team A is more likely to win."
DIFF_COLS = [c for c in X_full.columns if c.endswith("_diff")]
X = X_full[DIFF_COLS]

# 5. The answer we want the model to learn (the "target").
#    0 = team A won, 1 = draw, 2 = team B won.
def result_code(row):
    if row.winner == row.team_a:
        return 0
    if row.winner == "Draw":
        return 1
    return 2

y = matches.apply(result_code, axis=1)
LABELS = {0: "Team A win", 1: "Draw", 2: "Team B win"}

print("Feature table shape:", X.shape)
print("Example features for the first match:")
X.head(1).T.head(6)


### Step 2: The football maths for expected scores

In [ ]:
from math import exp, factorial

def poisson(k, mean):
    "Chance of exactly k goals when a team averages `mean` goals."
    return (mean ** k) * exp(-mean) / factorial(k)

def expected_scoreline(team_a, team_b, max_goals=8):
    """Predict goals for each team and the chance of each result.

    We blend a team's attack with the other team's defence. If A usually
    scores 2.0 and B usually lets in 1.4, A's expected goals sit between them.
    """
    a = stats_by_team.loc[team_a]
    b = stats_by_team.loc[team_b]
    xg_a = 0.5 * (a["avg_goals_scored"] + b["avg_goals_conceded"])
    xg_b = 0.5 * (b["avg_goals_scored"] + a["avg_goals_conceded"])

    p_a = p_draw = p_b = 0.0
    for ga in range(max_goals + 1):
        for gb in range(max_goals + 1):
            p = poisson(ga, xg_a) * poisson(gb, xg_b)
            if ga > gb:
                p_a += p
            elif ga == gb:
                p_draw += p
            else:
                p_b += p
    return {
        "expected_goals_a": round(xg_a, 2),
        "expected_goals_b": round(xg_b, 2),
        "predicted_score": f"{round(xg_a)} - {round(xg_b)}",
        "score_win_prob_a": round(p_a, 3),
        "score_draw_prob": round(p_draw, 3),
        "score_win_prob_b": round(p_b, 3),
    }


### Step 3: Split into practice and test sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print("Practice matches:", len(X_train), "| Test matches:", len(X_test))

### Step 4: Train the neural network

Neural networks care a lot about scale, so we scale the stats first. Two hidden layers of switches (32 then 16) is plenty for a dataset this size.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1500,
                  random_state=42, early_stopping=True))
model.fit(X_train, y_train)
print("Neural network trained.")

### Step 5: How good is it?

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

pred = model.predict(X_test)
baseline = y_test.value_counts(normalize=True).max()
print("Accuracy on hidden matches:", round(accuracy_score(y_test, pred), 3))
print("For comparison, always guessing the most common result gets:", round(baseline, 3))
print("Pure random guessing between three results gets about 0.333.")
print()
print(classification_report(y_test, pred,
      target_names=["Team A win", "Draw", "Team B win"]))

### Step 6: Predict the upcoming World Cup matches

In [ ]:
# Predict every upcoming fixture.
# For each match we report TWO things:
#   1. The model's win / draw / loss chances (from what it learned).
#   2. The expected scoreline from the football maths above.
report = []
for m in upcoming.itertuples():
    feats = pd.DataFrame([build_features(m.team_a, m.team_b)])[X.columns]
    probs = model.predict_proba(feats)[0]     # [P(A win), P(draw), P(B win)]
    score = expected_scoreline(m.team_a, m.team_b)
    report.append({
        "match": f"{m.team_a} vs {m.team_b}",
        "stage": m.stage,
        f"{m.team_a} win %": round(probs[0] * 100, 1),
        "draw %": round(probs[1] * 100, 1),
        f"{m.team_b} win %": round(probs[2] * 100, 1),
        "predicted score": score["predicted_score"],
        "expected goals": f'{score["expected_goals_a"]} - {score["expected_goals_b"]}',
    })

# Because each match has different team names, show them one at a time.
for row in report:
    print(row["match"], "  (" + row["stage"] + ")")
    for k, v in row.items():
        if k in ("match", "stage"):
            continue
        print(f"   {k:22s} {v}")
    print()


### Step 7: How sure is the network?

Unlike a tree, we cannot draw a neat flowchart. But we can still ask how confident it is. A confident prediction has one probability close to 100 percent.

In [ ]:
for m in upcoming.itertuples():
    feats = pd.DataFrame([build_features(m.team_a, m.team_b)])[X.columns]
    probs = model.predict_proba(feats)[0]
    top = max(probs)
    print(f"{m.team_a} vs {m.team_b}: most likely result has {round(top*100,1)}% confidence")

### Your turn

1. Change `hidden_layer_sizes=(32, 16)` to `(8,)`. A smaller brain. Does accuracy drop?
2. Try `(64, 32, 16)`. A bigger brain is not always better. Compare the test accuracy.
3. In your own words, one reason a coach might still prefer the decision tree from the last notebook.

## Submit your prediction

You have trained a model and predicted real fixtures. Now enter your answers
in the Week 2 quiz and prediction page:

**https://beagenius.org/tgp-2026teens-week2/06-quiz-and-predict.html**

Bring three things with you for a match of your choice:

1. The win probability for each team.
2. The predicted scoreline (the expected goals rounded to whole numbers).
3. One sentence on which stat mattered most, in your own words.

Good luck, and trust your numbers.
